# Experiment Reproduction:

*Multi-agent Deep Reinforcement Learning collaborative Traffic
Signal Control method considering intersection heterogeneity*
- Yiming Bie a
- Yuting Ji a
- Dongfang Ma

In [43]:
# Initialize by cloning the repository from GitHub
import os
repo_url = "https://github.com/IsaacFayle-Waters/MARL-TSC-SUMO-Group.git"
if not os.path.exists('MARL-TSC-SUMO-Group'):
    !git clone {repo_url}
else:
    print("Repository already exists. Use git pull if you need updates.")

Repository already exists. Use git pull if you need updates.


In [ ]:
#!unzip traffic_marl_project.zip

# 1. Environment & Dependency Setup
*Installing SUMO, PettingZoo, and configuring system paths.*

In [3]:
!apt-get update && apt-get install -y sumo sumo-tools
!pip install traci pettingzoo sumolib torch torchvision matplotlib

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,930 kB]
Get:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,669 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]


In [4]:
import os
os.environ["SUMO_HOME"] = "/usr/share/sumo"

In [5]:
import sys
import os
# Point to the root where the unzipped 'env' and 'agents' folders are
sys.path.append('/content/')

import traci

# 2. Traffic Network Generation
*Using netgenerate and randomTrips to create the 3x3 grid and traffic flows.*

In [7]:
!netgenerate \
--grid \
--grid.number=3 \
--tls.guess true \
--default.lanenumber=3 \
--grid.length=500 \
--default.speed=16.7 \
-o /content/sumo/network.net.xml

Success.


In [20]:
!python /usr/share/sumo/tools/randomTrips.py -n /content/sumo/network.net.xml -r /content/sumo/routes.rou.xml --flows 1000 --seed 42

calling /usr/share/sumo/bin/duarouter -n /content/sumo/network.net.xml -r trips.trips.xml --ignore-errors --begin 0 --end 3600 --no-step-log --no-warnings -o /content/sumo/routes.rou.xml
Success.


# 3. Multi-Agent Environment Definition
*The custom PettingZoo wrapper for the SUMO simulation.*

In [21]:
import traci
import os

# Diagnostic script to see what SUMO named your intersections
try:
    # Use a simpler start command just to check IDs
    traci.start(["sumo", "-c", "/content/sumo/config.sumocfg"])
    tls_ids = traci.trafficlight.getIDList()
    print(f"Detected Traffic Light IDs in SUMO: {tls_ids}")
    traci.close()
except Exception as e:
    print(f"Error during ID check: {e}")
    if "default" in traci.connection._connections: del traci.connection._connections["default"]

Error during ID check: Connection 'default' is already active.


# 4. Deep Q-Network Training
*Implementing the DQN Agent, Replay Buffer, and training loop.*

### Training Cell

In [48]:
import sys
import importlib
import env.traffic_env
import utils.metrics
import agents.replay_buffer
importlib.reload(env.traffic_env)
importlib.reload(utils.metrics)
importlib.reload(agents.replay_buffer)

from env.traffic_env import TrafficEnv
from utils.metrics import average_delay
from agents.replay_buffer import ReplayBuffer
from agents.dqn import DQN

import torch
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import random

def train():
    """
    Main Training Loop: Deep Q-Learning with Experience Replay.

    The goal is to learn a policy Pi that maps states to actions to maximize
    long-term discounted reward: G_t = sum(gamma^k * R_{t+k+1}).
    """
    env = TrafficEnv()
    state_size = 6    # [Density_L1, Queue_L1, Density_L2, Queue_L2, Density_L3, Queue_L3]
    action_size = 4   # Indices corresponding to SUMO TL Phases [0, 1, 2, 3]
    batch_size = 32   # Number of experiences sampled for each SGD update
    gamma = 0.95      # Discount factor: prioritizes immediate vs future rewards

    # The Q-Network: Approximates Q(s, a), the expected return of taking action 'a' in state 's'
    dqn_agent = DQN(state_size, action_size)
    optimizer = optim.Adam(dqn_agent.parameters(), lr=1e-4)

    # The Replay Buffer: Mitigates the problem of correlated data and non-stationary distributions
    memory = ReplayBuffer(size=10000)

    for episode in range(50):
        observations = env.reset()
        total_episode_reward = 0.0

        if episode % 10 == 0:
            print(f"\n--- Episode {episode + 1}/50 ---")

        for step in range(1000):
            actions = {}

            # --- 1. ACTION SELECTION (Exploration vs. Exploitation) ---
            for agent_id, obs in observations.items():
                obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
                with torch.no_grad():
                    q_values = dqn_agent(obs_tensor)

                # Epsilon-Greedy: 10% random exploration to discover new strategies
                if random.random() < 0.1:
                    action = random.randint(0, action_size - 1)
                else:
                    # Exploit: Select the action with the maximum predicted Q-value
                    action = q_values.argmax(dim=1).item()
                actions[agent_id] = action

            # --- 2. INTERACT WITH ENVIRONMENT ---
            # Apply actions to SUMO and advance simulation by one step
            next_obs, rewards, terminations, truncations, infos = env.step(actions)

            # --- 3. LEARN FROM EXPERIENCE ---
            for agent_id in observations.keys():
                # Cache the transition: (s_t, a_t, r_t, s_{t+1})
                transition = (observations[agent_id], actions[agent_id], rewards[agent_id], next_obs[agent_id])
                memory.push(transition)

                # Wait until buffer has enough data to perform a stable batch update
                if len(memory) > batch_size:
                    # Sample random transitions to break temporal correlation
                    batch = memory.sample(batch_size)
                    states, batch_actions, batch_rewards, next_states = zip(*batch)

                    # Vectorized conversion to PyTorch Tensors for GPU/CPU acceleration
                    states = torch.tensor(np.array(states), dtype=torch.float32)
                    batch_actions = torch.tensor(batch_actions, dtype=torch.long).unsqueeze(1)
                    batch_rewards = torch.tensor(batch_rewards, dtype=torch.float32).unsqueeze(1)
                    next_states = torch.tensor(np.array(next_states), dtype=torch.float32)

                    # Predicted Q(s_t, a_t): The value the network currently thinks this action is worth
                    current_q = dqn_agent(states).gather(1, batch_actions)

                    # Target Q using Bellman Optimality: R_t + gamma * max[Q(s_{t+1}, a')]
                    # We detach() the target to prevent gradients from flowing into the target calculation
                    max_next_q = dqn_agent(next_states).detach().max(1)[0].unsqueeze(1)
                    expected_q = batch_rewards + (gamma * max_next_q)

                    # Loss: Mean Squared Error between current prediction and Bellman target
                    loss = F.mse_loss(current_q, expected_q)

                    # Backpropagation: Adjust weights to minimize the Temporal Difference (TD) error
                    optimizer.zero_grad()
                    loss.backward()
                    optimizer.step()

            total_episode_reward += sum(rewards.values())
            observations = next_obs

        # Performance Logging
        if episode % 10 == 0:
            delay = average_delay()
            print(f"  Total Episode Reward: {total_episode_reward:.2f}")
            print(f"  Average Vehicle Delay at end of episode: {delay:.2f}s")

if __name__ == "__main__":
    train()

 Retrying in 1 seconds

--- Episode 1/50 ---
  Total Episode Reward: 130345.82
  Average Vehicle Delay at end of episode: 37.44s
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds

--- Episode 11/50 ---
  Total Episode Reward: 195170.90
  Average Vehicle Delay at end of episode: 39.33s
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds

--- Episode 21/50 ---
  Total Episode Reward: 189989.69
  Average Vehicle Delay at end of episode: 22.13s
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 1 seconds
 Retrying in 

# 5. Persistence & GitHub Sync
*Backing up files to Google Drive and pushing to the repository.*

In [28]:
!git clone https://github.com/IsaacFayle-Waters/MARL-TSC-SUMO-Group.git

fatal: destination path 'MARL-TSC-SUMO-Group' already exists and is not an empty directory.


### Syncing to GitHub
After cloning, you can copy your updated files into the repository folder before pushing:

```python
# 1. Copy files from Drive to the repo folder
!cp -r "/content/drive/MyDrive/Uni-Masters/Group Planning/MARL TSC/." /content/MARL-TSC-SUMO-Group/

# 2. Commit and Push
%cd /content/MARL-TSC-SUMO-Group/
!git config --global user.email "your_email@example.com"
!git config --global user.name "Your Name"
!git add .
!git commit -m "Update from Colab with improved TrafficEnv metrics"
# Note: You will need your GitHub Personal Access Token for the push command
# !git push origin main
```

In [29]:
!cp -r "/content/drive/MyDrive/Uni-Masters/Group Planning/MARL TSC/." /content/MARL-TSC-SUMO-Group/


In [30]:
%cd /content/MARL-TSC-SUMO-Group/
!git config --global user.email "zakfayle@gmail.com"
!git config --global user.name "Isaac Fayle-Waters"
!git add .
!git commit -m "Update from Colab with improved TrafficEnv metrics"


/content/MARL-TSC-SUMO-Group
[main 6299179] Update from Colab with improved TrafficEnv metrics
 16 files changed, 33209 insertions(+)
 create mode 100644 agents/__pycache__/dqn.cpython-312.pyc
 create mode 100644 agents/__pycache__/replay_buffer.cpython-312.pyc
 create mode 100644 agents/dqn.py
 create mode 100644 agents/replay_buffer.py
 create mode 100644 agents/train.py
 create mode 100644 env/__pycache__/traffic_env.cpython-312.pyc
 create mode 100644 env/observation.py
 create mode 100644 env/traffic_env.py
 create mode 100644 extracted_configuration.json
 create mode 100644 sumo/config.sumocfg
 create mode 100644 sumo/network.net.xml
 create mode 100644 sumo/routes.rou.alt.xml
 create mode 100644 sumo/routes.rou.xml
 create mode 100644 traffic_marl_project.zip
 create mode 100644 utils/__pycache__/metrics.cpython-312.pyc
 create mode 100644 utils/metrics.py


In [ ]:
!git push origin main

In [50]:
from google.colab import userdata
import os

# 1. Sync latest local files AND the notebook to the repo folder
!cp -r /content/env /content/sumo /content/agents /content/utils /content/extracted_configuration.json /content/MARL-TSC-SUMO-Group/

# Try to find and copy the current notebook file (.ipynb)
!find /content -maxdepth 1 -name "*.ipynb" -exec cp {} /content/MARL-TSC-SUMO-Group/ \;

# 2. Get the token from Colab Secrets
token = userdata.get('GH_TOKEN')
repo_name = "IsaacFayle-Waters/MARL-TSC-SUMO-Group"

# 3. Configure and Push
%cd /content/MARL-TSC-SUMO-Group/
!git add .
!git commit -m "Final commit: Including updated notebook and comprehensive source documentation"
!git remote set-url origin https://{token}@github.com/{repo_name}.git
!git push origin main

/content/MARL-TSC-SUMO-Group
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date


# Writefiles for Faster updating and reference

In [37]:
%%writefile /content/agents/dqn.py
"""
Deep Q-Network (DQN) Implementation.
This model approximates the Q-value function for traffic signal control decisions.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F

class DQN(nn.Module):
    def __init__(self, state_size, action_size):
        """
        Initialize the Neural Network.
        Args:
            state_size (int): Number of input features (Density/Queuing metrics).
            action_size (int): Number of possible traffic light phases.
        """
        super(DQN, self).__init__()
        # Standard multi-layer perceptron architecture
        self.fc1 = nn.Linear(state_size, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, action_size)

    def forward(self, x):
        """
        Forward pass to predict Q-values for each action.
        """
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

Overwriting /content/agents/dqn.py


In [38]:
%%writefile /content/agents/replay_buffer.py
"""
Replay Buffer for Experience Replay.
Stores past transitions to break temporal correlation during training.
"""
import random
from collections import deque

class ReplayBuffer:
    def __init__(self, size=10000):
        """
        Initialize a circular buffer using a deque.
        """
        self.buffer = deque(maxlen=size)

    def push(self, transition):
        """
        Add a new experience (s, a, r, s') to the buffer.
        """
        self.buffer.append(transition)

    def sample(self, batch_size):
        """
        Randomly sample a batch of experiences for stochastic gradient descent.
        """
        return random.sample(self.buffer, batch_size)

    def __len__(self):
        return len(self.buffer)

Overwriting /content/agents/replay_buffer.py


In [39]:
%%writefile /content/utils/metrics.py
"""
Utility functions for calculating traffic performance metrics via TraCI.
"""
import traci

def average_delay():
    """
    Calculates the average waiting time (delay) across all vehicles currently in the simulation.
    Waiting time is defined by SUMO as speed < 0.1 m/s.
    """
    veh_ids = traci.vehicle.getIDList()

    if len(veh_ids) == 0:
        return 0

    total = 0
    for v in veh_ids:
        total += traci.vehicle.getWaitingTime(v)

    return total / len(veh_ids)

Overwriting /content/utils/metrics.py


### 📝 TODO List for Experiment Alignment

- [ ] **Action Space**: Transition from discrete phase switching to 'Phase Duration Adjustment' (delta_t adjustment).
- [ ] **State Space**: Expand observations to include 'Remaining Exit Space' and 'Global Network Metrics'.
- [ ] **Network Scale**: Increase grid from 3x3 (9 agents) to 5x5 (25 agents) to match the paper's scenario.
- [ ] **Reward Tuning**: Implement the 'Heterogeneous Correlation Index' and calibrate weights for delay vs. throughput.
- [ ] **Simulation Length**: Increase episode duration to 3600 seconds (1 full hour).
- [ ] **Hyperparameters**: Verify 'Scaling Factor Q' and 'Discount Factor (0.95)' across all training scripts.

In [35]:
readme_content = """# MARL-TSC-SUMO-Group

## Project Purpose
This project is an experiment reproduction of the **'Multi-agent Deep Reinforcement Learning collaborative Traffic Signal Control method considering intersection heterogeneity'** (MARL SGAT).

## Objective
The goal is to maximize long-term cumulative reward (minimizing delay and maximizing throughput) across a road network using a 5x5 grid of 25 intersections controlled by decentralized DQN agents.

## Current Status
- Environment: PettingZoo-compatible wrapper for SUMO.
- Agent: Deep Q-Network (DQN) with Replay Buffer.
- Metrics: Average vehicle delay and throughput-based rewards.
"""
with open('/content/MARL-TSC-SUMO-Group/README.md', 'w') as f:
    f.write(readme_content)
print('README updated.')

README updated.


In [44]:
# Cell merged with ae9d185d to remove redundancy.

In [45]:
%%writefile /content/env/traffic_env.py
"""
TrafficEnv: A Multi-Agent Reinforcement Learning environment using SUMO.
Aligns with MARL SGAT experiment parameters for state and reward definitions.
"""
from pettingzoo import ParallelEnv
import traci
import traci.connection
import numpy as np

class TrafficEnv(ParallelEnv):
    metadata = {"name": "traffic_marl_env"}

    def __init__(self):
        self.sumo_ids = ['A1', 'B0', 'B1', 'B2', 'C1']
        self.agents = [f"agent_{i}" for i in range(len(self.sumo_ids))]
        self.agent_to_sumo = {f"agent_{i}": sid for i, sid in enumerate(self.sumo_ids)}
        self.last_waiting_times = {agent: 0 for agent in self.agents}

    def reset(self, seed=None, options=None):
        try:
            if "default" in traci.connection._connections: traci.close()
        except Exception: pass
        finally:
            if "default" in traci.connection._connections: del traci.connection._connections["default"]

        traci.start(["sumo", "-c", "/content/sumo/config.sumocfg"])
        self.last_waiting_times = {agent: 0 for agent in self.agents}
        observations = {agent: self._get_obs(agent) for agent in self.agents}
        return observations

    def step(self, actions):
        for agent, action in actions.items():
            self._apply_action(agent, action)
        traci.simulationStep()

        observations = {agent: self._get_obs(agent) for agent in self.agents}
        rewards = {agent: self._compute_reward(agent) for agent in self.agents}
        terminations = {agent: False for agent in self.agents}
        truncations = {agent: False for agent in self.agents}
        infos = {agent: {} for agent in self.agents}
        return observations, rewards, terminations, truncations, infos

    def _get_obs(self, agent):
        sumo_id = self.agent_to_sumo[agent]
        lanes = traci.trafficlight.getControlledLanes(sumo_id)
        unique_lanes = list(dict.fromkeys(lanes))
        obs = []
        for lane in unique_lanes[:3]:
            length = traci.lane.getLength(lane)
            density = traci.lane.getLastStepVehicleNumber(lane) / length
            queuing = traci.lane.getLastStepHaltingNumber(lane) / length
            obs.extend([density, queuing])
        while len(obs) < 6: obs.append(0.0)
        return np.array(obs, dtype=np.float32)

    def _apply_action(self, agent, action):
        sumo_id = self.agent_to_sumo[agent]
        phases = [0, 1, 2, 3]
        traci.trafficlight.setPhase(sumo_id, phases[action % len(phases)])

    def _compute_reward(self, agent):
        sumo_id = self.agent_to_sumo[agent]
        lanes = traci.trafficlight.getControlledLanes(sumo_id)
        current_waiting_time = sum([traci.lane.getWaitingTime(l) for l in lanes])
        throughput = sum([traci.lane.getLastStepVehicleNumber(l) for l in lanes])
        delay_component = (self.last_waiting_times[agent] - current_waiting_time) / 100.0
        reward = delay_component + (throughput * 0.1)
        self.last_waiting_times[agent] = current_waiting_time
        return float(reward)

Overwriting /content/env/traffic_env.py


In [46]:
import torch
from env.traffic_env import TrafficEnv
from agents.dqn import DQN
import traci

def observe_policy_behavior(steps=20):
    """
    Runs a short simulation to print the real-time decisions of the agent.
    Useful for verifying that the agent isn't 'stuck' in a specific phase.
    """
    env = TrafficEnv()
    state_size = 6
    action_size = 4

    # Initialize and set to evaluation mode (no weight updates)
    agent = DQN(state_size, action_size)
    agent.eval()

    obs = env.reset()
    print(f"{'Step':<5} | {'Agent':<8} | {'Action':<7} | {'SUMO Phase State'}")
    print("-" * 45)

    for s in range(steps):
        actions = {}
        for agent_id, o in obs.items():
            # Convert observation to tensor for the Neural Network
            o_tensor = torch.tensor(o, dtype=torch.float32).unsqueeze(0)
            with torch.no_grad():
                # Get the action with the highest predicted Q-value
                action = agent(o_tensor).argmax(dim=1).item()
            actions[agent_id] = action

            # Fetch the actual 'G/r/y' string from the SUMO simulation core
            sumo_id = env.agent_to_sumo[agent_id]
            phase_state = traci.trafficlight.getRedYellowGreenState(sumo_id)

            # Print periodic updates to track behavior
            if s % 5 == 0:
                print(f"{s:<5} | {agent_id:<8} | {action:<7} | {phase_state}")

        # Step the environment with the chosen actions
        obs, _, _, _, _ = env.step(actions)

    traci.close()

if __name__ == '__main__':
    observe_policy_behavior()

 Retrying in 1 seconds
Step  | Agent    | Action  | SUMO Phase State
---------------------------------------------
0     | agent_0  | 2       | GGGgggrrrrrGGGGg
0     | agent_1  | 2       | rrrrrGGGGgGGGggg
0     | agent_2  | 2       | GGGGggrrrrrrGGGGggrrrrrr
0     | agent_3  | 2       | GGGgggrrrrrGGGGg
0     | agent_4  | 2       | GGGGgGGGgggrrrrr
5     | agent_0  | 2       | rrrrrrGGGGgGrrrr
5     | agent_1  | 2       | GGGGgGrrrrrrrrrr
5     | agent_2  | 2       | rrrrrrGGGGggrrrrrrGGGGgg
5     | agent_3  | 2       | rrrrrrGGGGgGrrrr
5     | agent_4  | 2       | GrrrrrrrrrrGGGGg
10    | agent_0  | 2       | rrrrrrGGGGgGrrrr
10    | agent_1  | 2       | GGGGgGrrrrrrrrrr
10    | agent_2  | 2       | rrrrrrGGGGggrrrrrrGGGGgg
10    | agent_3  | 2       | rrrrrrGGGGgGrrrr
10    | agent_4  | 2       | GrrrrrrrrrrGGGGg
15    | agent_0  | 2       | rrrrrrGGGGgGrrrr
15    | agent_1  | 2       | GGGGgGrrrrrrrrrr
15    | agent_2  | 2       | rrrrrrGGGGggrrrrrrGGGGgg
15    | agent_3  | 2     

In [26]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Updated path based on your specific Uni directory
project_path = '/content/drive/MyDrive/Uni-Masters/Group Planning/MARL TSC'

if not os.path.exists(project_path):
    os.makedirs(project_path)
    print(f'Created directory: {project_path}')

# Execute this to sync your current work to Google Drive
!cp -r /content/env /content/sumo /content/agents /content/utils /content/extracted_configuration.json "{project_path}"
print(f'Files successfully backed up to {project_path}')

Mounted at /content/drive
Files successfully backed up to /content/drive/MyDrive/Uni-Masters/Group Planning/MARL TSC


### GitHub Persistence
To connect this project to GitHub from Colab:

1. **Generate a Personal Access Token (PAT)** on GitHub.
2. **Initialize Git** in your project folder (e.g., in `/content/drive/MyDrive/traffic_marl_project`).
3. **Configure your identity** and push using the PAT as the password.

```python
# Example for pushing
!git config --global user.email "your_email@example.com"
!git config --global user.name "Your Name"
!git remote add origin https://<TOKEN>@github.com/<USERNAME>/<REPO>.git
!git add .
!git commit -m "Sync from Colab"
!git push -u origin main
```

In [40]:
# Cell removed to reduce redundancy - config is loaded in config_data variable

In [41]:
# Cell removed to reduce redundancy - replay_buffer.py is already defined

In [42]:
# Cell removed to reduce redundancy - metrics.py is already defined

In [14]:
import torch
from env.traffic_env import TrafficEnv
from agents.dqn import DQN
import numpy as np

def evaluate(model_path=None):
    env = TrafficEnv()
    state_size = 6
    action_size = 4

    # Initialize agent
    eval_agent = DQN(state_size, action_size)

    # If you saved a model, load it here:
    # eval_agent.load_state_dict(torch.load(model_path))
    eval_agent.eval() # Set to evaluation mode

    observations = env.reset()
    total_reward = 0
    steps = 0

    print("Starting Evaluation...")

    for step in range(300):
        actions = {}
        for agent_id, obs in observations.items():
            obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)
            with torch.no_grad():
                q_values = eval_agent(obs_tensor)
            # Take the best action (greedy)
            action = q_values.argmax(dim=1).item()
            actions[agent_id] = action

        next_obs, rewards, terms, truncs, infos = env.step(actions)
        total_reward += sum(rewards.values())
        observations = next_obs
        steps += 1

    print(f"Evaluation Finished.")
    print(f"Total Reward: {total_reward:.2f}")
    print(f"Steps: {steps}")

if __name__ == '__main__':
    evaluate()

 Retrying in 1 seconds
Starting Evaluation...
Evaluation Finished.
Total Reward: 0.00
Steps: 300


#### LLM USAGE:
##### Main Extraction and Files
From ChatGPT --> https://chatgpt.com/share/69b57d24-5940-8011-a264-6d93f952b33b
#### Assistance From Colab's Inbuilt Gemini Instances
- Gemini 2.5 Flash
- Gemini 3 Flash


Editing and ensuring reproducable environment.
